# [SQL 실습 #11] MySQL HAVING 기초
## GROUP BY 결과에 조건 걸기

> **학생용 실습 노트북 - TODO + 자동점검형**

이번 실습은 **SQL-11.pdf**의 흐름에 맞추어 구성했습니다.

지난 시간에는 `GROUP BY`를 이용해 학년별·학과별로 데이터를 묶어 집계했습니다.  
이번 시간에는 그룹으로 묶은 결과에 조건을 거는 `HAVING`을 배웁니다.

---

## 오늘의 학습 목표

1. `HAVING`의 역할을 설명할 수 있다.
2. `WHERE`와 `HAVING`의 차이를 설명할 수 있다.
3. 평균 점수가 일정 기준 이상인 학과를 조회할 수 있다.
4. 학생 수가 일정 기준 이상인 학과를 조회할 수 있다.
5. 학년별 평균에 조건을 걸 수 있다.
6. `HAVING + ORDER BY`를 함께 사용할 수 있다.
7. `WHERE + GROUP BY + HAVING`을 함께 사용할 수 있다.
8. `HAVING COUNT(*)`와 `HAVING COUNT(score)`의 차이를 이해한다.
9. `HAVING SUM()`으로 합계 조건을 걸 수 있다.
10. `HAVING MAX()`로 최고 점수 조건을 걸 수 있다.

### 오늘의 핵심

> **WHERE는 그룹으로 묶기 전에 조건을 걸고, HAVING은 GROUP BY로 묶은 결과에 조건을 겁니다.**


# 1. MySQL 실습 환경 준비

Google Colab에서 사용할 수 있도록 MySQL 서버를 설치하고 실행합니다.

> 아래 셀은 수정하지 말고 실행하세요.


In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start
print("✅ MySQL 서버 준비 완료")


# 2. 실습 데이터 준비

SQL-11 자료의 예제와 같은 흐름으로 `students` 테이블을 준비합니다.

| id | name | grade | class_name | score |
|---:|---|---:|---|---:|
| 1 | 김부장 | 3 | 컴퓨터과 | 85 |
| 2 | 훈이 | 1 | 영훈반 | 90 |
| 3 | 라즈베리 | 2 | 임베디드반 | 78 |
| 4 | 리눅스 | 3 | 서버반 | 88 |
| 5 | 파이썬 | 1 | 프로그래밍반 | 95 |
| 6 | 김코딩 | 1 | 컴퓨터과 | 82 |
| 7 | 데이터왕 | 2 | 데이터베이스반 | 91 |
| 8 | 홍길동 | 2 | 컴퓨터과 | 76 |
| 9 | 이몽룡 | 3 | 데이터베이스반 | 89 |
| 10 | 정렬이 | 1 | SQL반 | NULL |
| 11 | 제한이 | 2 | SQL반 | 84 |
| 12 | 그룹이 | 3 | 컴퓨터과 | 92 |

`정렬이` 학생의 `score`는 `NULL`입니다.  
뒤에서 `COUNT(*)`와 `COUNT(score)`의 차이를 확인할 때 사용합니다.


In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;
DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    score INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, score) VALUES
('김부장', 3, '컴퓨터과', 85),
('훈이', 1, '영훈반', 90),
('라즈베리', 2, '임베디드반', 78),
('리눅스', 3, '서버반', 88),
('파이썬', 1, '프로그래밍반', 95),
('김코딩', 1, '컴퓨터과', 82),
('데이터왕', 2, '데이터베이스반', 91),
('홍길동', 2, '컴퓨터과', 76),
('이몽룡', 3, '데이터베이스반', 89),
('정렬이', 1, 'SQL반', NULL),
('제한이', 2, 'SQL반', 84),
('그룹이', 3, '컴퓨터과', 92);

SELECT id, name, grade, class_name, score
FROM students
ORDER BY id;
SQL


# 3. SQL 실행 / 자동점검 도우미

- SQL에 `TODO`가 남아 있으면 실행하지 않습니다.
- 학생 실습 셀은 조회 전용입니다.
- 자동점검은 결과값과 순서를 비교합니다.

> 아래 셀은 수정하지 마세요.


In [ ]:
import subprocess, textwrap, re
TODO_STATUS = {}

def _mysql(sql, no_header=False):
    cmd = ["sudo", "mysql", "--batch", "--raw"]
    if no_header:
        cmd.append("-N")
    p = subprocess.run(
        cmd,
        input=textwrap.dedent(sql).strip(),
        text=True,
        capture_output=True
    )
    return p.returncode == 0, p.stdout.strip(), p.stderr.strip()

def _clean(sql):
    return "\n".join(
        line for line in textwrap.dedent(sql).splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ).strip()

def run_sql(sql):
    real = _clean(sql)

    if not real:
        print("⚠️ SQL이 비어 있습니다.")
        return False

    if "TODO" in real:
        print("⚠️ TODO가 남아 있습니다.")
        return False

    if re.search(r"\b(INSERT|UPDATE|DELETE|DROP|TRUNCATE|ALTER|CREATE)\b", real, re.I):
        print("❌ TODO 셀은 조회 전용입니다.")
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)

    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False

    print("✅ 조회 완료")
    return True

def _result(sql):
    ok, out, err = _mysql(sql, True)
    if not ok:
        raise RuntimeError(err)
    return out.splitlines() if out else []

def check_todo(n, student_sql, answer_sql):
    real = _clean(student_sql)

    if "TODO" in real:
        TODO_STATUS[n] = False
        print(f"🔎 TODO {n}: ❌ 먼저 TODO를 완성하세요.")
        return False

    try:
        actual = _result(student_sql)
        expected = _result(answer_sql)
        passed = actual == expected
    except Exception as e:
        TODO_STATUS[n] = False
        print(f"🔎 TODO {n}: ❌ SQL 오류 - {e}")
        return False

    TODO_STATUS[n] = passed
    print(f"🔎 TODO {n}: " + ("✅ 통과" if passed else "❌ 다시 확인"))

    if not passed:
        print("예상:", expected)
        print("현재:", actual)

    return passed

def show_progress(total):
    print("===== TODO 진행 현황 =====")
    passed = 0

    for i in range(1, total + 1):
        state = TODO_STATUS.get(i)
        mark = "✅" if state is True else ("❌" if state is False else "⬜")
        if state is True:
            passed += 1
        print(f"{mark} TODO {i}")

    print(f"\n진행률: {passed}/{total}")


# 4. HAVING이란?

지난 시간에는 학과별 평균 점수를 다음처럼 구했습니다.

```sql
SELECT
    class_name,
    ROUND(AVG(score), 1) AS 평균점수
FROM students
GROUP BY class_name;
```

여기에서 **평균이 85점 이상인 학과만** 보고 싶다면 `HAVING`을 사용합니다.

```sql
SELECT
    class_name,
    ROUND(AVG(score), 1) AS 평균점수
FROM students
GROUP BY class_name
HAVING AVG(score) >= 85;
```

> **HAVING은 집계 결과에 조건을 걸 때 사용하는 문법입니다.**


# 5. WHERE와 HAVING의 차이

두 문법은 모두 조건을 걸지만 사용 시점이 다릅니다.

### WHERE
개별 행을 먼저 걸러냅니다.

```sql
SELECT *
FROM students
WHERE score >= 80;
```

### HAVING
`GROUP BY`로 만든 그룹 결과를 걸러냅니다.

```sql
SELECT
    class_name,
    AVG(score)
FROM students
GROUP BY class_name
HAVING AVG(score) >= 85;
```

### 쉽게 기억하기

> **WHERE는 바구니에 담기 전에 골라내는 것**  
> **HAVING은 바구니에 담은 뒤 결과를 보고 골라내는 것**


# 6. 평균 점수가 85점 이상인 학과만 조회하기

## TODO 1

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY class_name ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    1,
    sql,
    'SELECT class_name, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY class_name HAVING AVG(score) >= 85 ORDER BY class_name ASC;'
)


# 7. 학생 수가 2명 이상인 학과만 조회하기

## TODO 2

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY class_name ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    2,
    sql,
    'SELECT class_name, COUNT(*) AS 학생수 FROM students GROUP BY class_name HAVING COUNT(*) >= 2 ORDER BY class_name ASC;'
)


# 8. 학년별 평균 점수가 85점 이상인 학년 조회하기

## TODO 3

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY grade ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    3,
    sql,
    'SELECT grade, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY grade HAVING AVG(score) >= 85 ORDER BY grade ASC;'
)


# 9. HAVING과 ORDER BY 함께 사용하기

## TODO 4

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 평균점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY TODO DESC;
'''
run_sql(sql)


In [ ]:
check_todo(
    4,
    sql,
    'SELECT class_name, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY class_name HAVING AVG(score) >= 85 ORDER BY 평균점수 DESC;'
)


# 10. WHERE + GROUP BY + HAVING 함께 사용하기

## TODO 5

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수,
    TODO AS 평균점수
FROM students
WHERE TODO
GROUP BY TODO
HAVING TODO
ORDER BY 평균점수 DESC;
'''
run_sql(sql)


In [ ]:
check_todo(
    5,
    sql,
    'SELECT class_name, COUNT(*) AS 학생수, ROUND(AVG(score),1) AS 평균점수 FROM students WHERE score >= 80 GROUP BY class_name HAVING AVG(score) >= 85 ORDER BY 평균점수 DESC;'
)


# 11. HAVING COUNT(*)와 COUNT(score) 비교하기

## TODO 6

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 전체학생수,
    TODO AS 점수입력수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY class_name ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    6,
    sql,
    'SELECT class_name, COUNT(*) AS 전체학생수, COUNT(score) AS 점수입력수 FROM students GROUP BY class_name HAVING COUNT(score) >= 2 ORDER BY class_name ASC;'
)


# 12. 학년별 점수 합계가 300점 이상인 학년 조회하기

## TODO 7

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 점수합계
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY grade ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    7,
    sql,
    'SELECT grade, SUM(score) AS 점수합계 FROM students GROUP BY grade HAVING SUM(score) >= 300 ORDER BY grade ASC;'
)


# 13. 학과별 최고 점수가 90점 이상인 학과 조회하기

## TODO 8

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 최고점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY class_name ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    8,
    sql,
    'SELECT class_name, MAX(score) AS 최고점수 FROM students GROUP BY class_name HAVING MAX(score) >= 90 ORDER BY class_name ASC;'
)


# 14. 학과별 최저 점수가 80점 이상인 학과 조회하기

## TODO 9

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 최저점수
FROM students
GROUP BY TODO
HAVING TODO
ORDER BY class_name ASC;
'''
run_sql(sql)


In [ ]:
check_todo(
    9,
    sql,
    'SELECT class_name, MIN(score) AS 최저점수 FROM students GROUP BY class_name HAVING MIN(score) >= 80 ORDER BY class_name ASC;'
)


# 15. 학생 수 2명 이상이면서 평균 80점 이상인 학과 조회하기

## TODO 10

아래 `TODO`를 완성하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO,
    TODO AS 학생수,
    TODO AS 평균점수
FROM students
GROUP BY TODO
HAVING TODO
   AND TODO
ORDER BY 평균점수 DESC;
'''
run_sql(sql)


In [ ]:
check_todo(
    10,
    sql,
    'SELECT class_name, COUNT(*) AS 학생수, ROUND(AVG(score),1) AS 평균점수 FROM students GROUP BY class_name HAVING COUNT(*) >= 2 AND AVG(score) >= 80 ORDER BY 평균점수 DESC;'
)


# 16. HAVING에 별명을 사용할 수 있을까?

MySQL에서는 다음처럼 별명을 사용할 수도 있습니다.

```sql
SELECT
    class_name,
    ROUND(AVG(score), 1) AS 평균점수
FROM students
GROUP BY class_name
HAVING 평균점수 >= 85;
```

다만 학습 단계에서는 아래처럼 **집계 함수를 직접 쓰는 방식**이 더 명확합니다.

```sql
HAVING AVG(score) >= 85;
```

왜냐하면 `HAVING`이 무엇을 기준으로 판단하는지 바로 보이기 때문입니다.


# 17. COUNT(*)와 COUNT(score)의 차이

다음 두 식은 의미가 다릅니다.

```sql
COUNT(*)
```

→ 전체 행 개수

```sql
COUNT(score)
```

→ `score`가 NULL이 아닌 행 개수

따라서:

```sql
HAVING COUNT(*) >= 2
```

은 **전체 학생이 2명 이상인 그룹**을 의미하고,

```sql
HAVING COUNT(score) >= 2
```

는 **점수가 입력된 학생이 2명 이상인 그룹**을 의미합니다.


# 18. SQL 작성 순서 정리

`WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`를 함께 사용할 때 기본 순서는 다음과 같습니다.

```sql
SELECT 그룹기준컬럼, 집계함수
FROM 테이블명
WHERE 개별행조건
GROUP BY 그룹기준컬럼
HAVING 그룹조건
ORDER BY 정렬기준;
```

### 기억하기

```text
FROM
 ↓
WHERE
 ↓
GROUP BY
 ↓
HAVING
 ↓
ORDER BY
```

> **WHERE는 먼저 걸러내고, GROUP BY로 묶고, HAVING으로 결과를 다시 걸러냅니다.**


# 19. WHERE로 할 수 있는 조건 / HAVING으로 해야 하는 조건

### WHERE로 처리하기 좋은 조건

개별 행만 보면 판단할 수 있는 조건입니다.

```sql
WHERE grade = 2
WHERE score >= 80
WHERE class_name = '컴퓨터과'
```

### HAVING으로 처리해야 하는 조건

그룹별 계산 결과를 봐야 판단할 수 있는 조건입니다.

```sql
HAVING AVG(score) >= 85
HAVING COUNT(*) >= 2
HAVING SUM(score) >= 300
HAVING MAX(score) >= 90
```


# 20. 자주 발생하는 실수

### 1) 집계 함수 조건을 WHERE에 쓰는 경우

잘못된 예:

```sql
SELECT class_name, AVG(score)
FROM students
WHERE AVG(score) >= 85
GROUP BY class_name;
```

올바른 예:

```sql
SELECT class_name, AVG(score)
FROM students
GROUP BY class_name
HAVING AVG(score) >= 85;
```

---

### 2) HAVING을 GROUP BY 없이 무조건 쓰는 경우

`HAVING`은 보통 `GROUP BY`로 만든 결과에 조건을 걸 때 사용합니다.

---

### 3) WHERE와 HAVING의 순서를 바꾸는 경우

기본 순서:

```text
WHERE → GROUP BY → HAVING
```

---

### 4) COUNT(*)와 COUNT(score)를 구분하지 않는 경우

- `COUNT(*)` : 전체 행
- `COUNT(score)` : NULL이 아닌 score 값


# 21. 종합 실습

### 문제 A
학생 수가 2명 이상인 학과 중 평균 점수가 80점 이상인 학과를 조회하세요.

### 문제 B
점수가 80점 이상인 학생만 대상으로 학과별 평균을 구한 뒤, 평균이 85점 이상인 학과만 조회하세요.

### 문제 C
학년별 점수 합계가 300점 이상인 학년을 조회하세요.

### 문제 D
학과별 최고 점수가 90점 이상인 학과를 조회하세요.

### 문제 E
점수가 입력된 학생이 2명 이상인 학과를 조회하세요.

<details>
<summary>정답 예시 보기</summary>

```sql
-- A
SELECT class_name,
       COUNT(*) AS 학생수,
       ROUND(AVG(score),1) AS 평균점수
FROM students
GROUP BY class_name
HAVING COUNT(*) >= 2
   AND AVG(score) >= 80;

-- B
SELECT class_name,
       ROUND(AVG(score),1) AS 평균점수
FROM students
WHERE score >= 80
GROUP BY class_name
HAVING AVG(score) >= 85;

-- C
SELECT grade,
       SUM(score) AS 점수합계
FROM students
GROUP BY grade
HAVING SUM(score) >= 300;

-- D
SELECT class_name,
       MAX(score) AS 최고점수
FROM students
GROUP BY class_name
HAVING MAX(score) >= 90;

-- E
SELECT class_name,
       COUNT(score) AS 점수입력수
FROM students
GROUP BY class_name
HAVING COUNT(score) >= 2;
```

</details>


# 22. 오늘 배운 내용 정리

| 문법 | 역할 |
|---|---|
| `WHERE` | 그룹으로 묶기 전에 개별 행을 조건으로 선택 |
| `GROUP BY` | 같은 값을 가진 행을 그룹으로 묶기 |
| `HAVING` | 그룹별 집계 결과에 조건 적용 |
| `ORDER BY` | 최종 결과 정렬 |
| `COUNT(*)` | 전체 행 개수 |
| `COUNT(score)` | NULL이 아닌 score 개수 |
| `SUM()` | 합계 |
| `AVG()` | 평균 |
| `MAX()` | 최댓값 |
| `MIN()` | 최솟값 |

### 핵심 문장

> **WHERE는 바구니에 담기 전에 고르고, GROUP BY는 바구니로 묶고, HAVING은 바구니 결과를 보고 다시 고릅니다.**


# 23. 전체 TODO 진행 현황

In [ ]:
show_progress(10)

# 24. 자기 점검

- [ ] `HAVING`의 역할을 설명할 수 있다.
- [ ] `WHERE`와 `HAVING`의 차이를 설명할 수 있다.
- [ ] 평균 점수를 기준으로 그룹을 필터링할 수 있다.
- [ ] 학생 수를 기준으로 그룹을 필터링할 수 있다.
- [ ] `HAVING + ORDER BY`를 사용할 수 있다.
- [ ] `WHERE + GROUP BY + HAVING`을 함께 사용할 수 있다.
- [ ] `COUNT(*)`와 `COUNT(score)`의 차이를 설명할 수 있다.
- [ ] `HAVING SUM()`을 사용할 수 있다.
- [ ] `HAVING MAX()`를 사용할 수 있다.
- [ ] 여러 HAVING 조건을 `AND`로 연결할 수 있다.


# 다음 시간 예고

## [SQL 실습 #12] MySQL 기본키와 AUTO_INCREMENT 다시 이해하기

다음 시간에는 다음 개념을 다시 정리합니다.

- `PRIMARY KEY`
- `AUTO_INCREMENT`
- `NOT NULL`

처음에는 그냥 따라 입력했던 옵션들이 실제로 어떤 의미를 가지는지 자세히 살펴봅니다.
